[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C25_Long_Context_Course/03_sparse_attention/03_sparse_attention.ipynb)

# 03 · 稀疏与滑窗注意力（用 numpy 从零写出）

**算力墙**：FlashAttention 让 O(n²) 显存可行，但算力仍是 O(n²)。稀疏注意力让每个 token 只看部分 key，把复杂度降到 **O(n·W)**。

**路线**：
1. 掩码注意力的统一框架（一个布尔 Mask 决定算哪些连接）
2. **滑动窗口 SWA**：每 token 看最近 W 个 → 对拍全注意力的局部召回
3. 多层感受野：局部 + 深度 = 长程
4. **attention sink + StreamingLLM**：丢 sink 会崩、保 sink 救场
5. **block-sparse**：局部 + 全局列 的块掩码
6. **召回率**：量化稀疏漏掉了多少（针在窗内 vs 窗外）
7. ✏️ 练习（滑窗掩码 / sink 保留 / block-sparse / 召回率）→ 📖 答案 → 🧪 胶囊

> 本课纪律：稀疏是**近似**，每个稀疏模式都用召回率对拍全注意力，量化损失。

## 1 · 掩码注意力：统一框架

所有稀疏注意力 = 一个布尔 `mask`（`True`=保留该连接）作用于分数矩阵：保留处算分数，其余置 `-inf`（softmax 后权重 0）。
先写这个统一框架，后面各种稀疏只是换 `mask`。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def softmax_rows(S):
    m = S.max(axis=-1, keepdims=True)
    e = np.exp(S - m)
    return e / np.where(e.sum(-1, keepdims=True) == 0, 1.0, e.sum(-1, keepdims=True))

def masked_attention(Q, K, V, mask):
    '''mask: (n,n) bool, True=保留。返回注意力输出与概率矩阵 P。'''
    n, d = Q.shape
    S = Q @ K.T / np.sqrt(d)
    S = np.where(mask, S, -np.inf)            # 未保留处 -inf
    P = softmax_rows(S)
    return P @ V, P

n, d = 8, 4
Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
full_mask = np.ones((n,n), bool)             # 全注意力 = 全 True
O_full, P_full = masked_attention(Q, K, V, full_mask)
assert np.allclose(P_full.sum(1), 1.0), '每行概率和=1'
print('✅ 掩码注意力框架就绪；全 True 掩码 = 普通全注意力')

## 2 · 滑动窗口 SWA：每 token 看最近 W 个

因果滑窗：`mask[i,j] = (i-W < j <= i)`。复杂度 O(n·W)，线性于序列长。
对拍全注意力：滑窗只保留局部，远处连接被丢——这正是它省算力的代价。

In [ ]:
def sliding_window_mask(n, W, causal=True):
    '''每个 query i 看 (i-W, i] 的 key（含自己）。'''
    i = np.arange(n)[:, None]; j = np.arange(n)[None, :]
    m = (j <= i) & (j > i - W) if causal else (np.abs(i - j) < W)
    return m

n, W = 12, 4
sw = sliding_window_mask(n, W)
print('滑窗掩码每行保留的 key 数:', sw.sum(1))
assert sw[0].sum() == 1, '第0个token只看自己'
assert sw[-1].sum() == W, '靠后的 token 看满 W 个'
# 每行最多 W 个连接 -> O(n*W)
assert sw.sum() <= n * W, '总连接数 <= n*W（线性）'
full_conn = n * (n + 1) // 2                  # 因果全注意力的连接数
print(f'滑窗连接数 {sw.sum()} vs 因果全注意力 {full_conn}（省了 {1 - sw.sum()/full_conn:.0%}）')
print('✅ 滑窗 = 局部带状掩码，复杂度 O(n·W)')

## 3 · 多层感受野：局部 + 深度 = 长程

单层滑窗只看 W，但信息逐层接力，L 层后感受野 ≈ L×W。从零算这条扩张曲线，理解滑窗为何能建模长依赖。

In [ ]:
def receptive_field(W, n_layers):
    '''L 层滑窗(每层窗口W, 因果)后，一个 token 能间接看到多远(向前的 token 数)。'''
    # 每层向前延伸 (W-1)，L 层累积；+1 含自己
    return n_layers * (W - 1) + 1

W = 4096
print(f"{'层数':>6s} {'感受野(token)':>16s}")
for L in [1, 4, 16, 32]:
    print(f'{L:>6d} {receptive_field(W, L):>16d}')
rf32 = receptive_field(4096, 32)
assert rf32 > 100000, '32 层 W=4096 感受野应 >100k'
assert receptive_field(W, 2) > receptive_field(W, 1), '层数越多感受野越大'
print(f'\n✅ 32 层 × W=4096 → 感受野 ~{rf32} token（>128k）—— 局部注意力靠深度覆盖长程')

## 4 · attention sink + StreamingLLM：丢 sink 会崩

softmax 强制权重和=1，模型把多余注意力倾倒到最前面几个 token（sink）当泄压阀。
流式时若纯滑窗丢掉 sink，注意力分布失衡。我们构造有 sink 倾向的分数，对比「丢 sink」vs「保 sink」。

In [ ]:
def streaming_mask(n, W, n_sink):
    '''StreamingLLM: 保留最前 n_sink 个 sink + 最近 W 个（因果）。'''
    i = np.arange(n)[:, None]; j = np.arange(n)[None, :]
    local = (j <= i) & (j > i - W)
    sink  = (j < n_sink) & (j <= i)              # 前 n_sink 个始终保留
    return local | sink

n, W, n_sink = 64, 8, 4
# 构造「sink 倾向」：让所有 query 对前几个 key 的分数偏高（模拟真实模型的 sink 现象）
Q = rng.standard_normal((n, 4)); K = rng.standard_normal((n, 4)); V = rng.standard_normal((n, 4))
S = Q @ K.T / 2.0
S[:, :n_sink] += 5.0                              # 前 n_sink 个 key 是 sink（吸注意力）

def attn_from_scores(S, mask, V):
    Sm = np.where(mask, S, -np.inf)
    P = softmax_rows(Sm)
    return P @ V, P

pure_sw   = sliding_window_mask(n, W)            # 丢 sink
streaming = streaming_mask(n, W, n_sink)         # 保 sink
_, P_pure   = attn_from_scores(S, pure_sw, V)
_, P_stream = attn_from_scores(S, streaming, V)
# 看一个靠后的 query：它对 sink 的注意力是否被保住
q = n - 1
sink_mass_pure   = P_pure[q, :n_sink].sum()
sink_mass_stream = P_stream[q, :n_sink].sum()
print(f'靠后 query 对 sink 的注意力质量：纯滑窗={sink_mass_pure:.3f}  StreamingLLM={sink_mass_stream:.3f}')
assert sink_mass_pure < 0.01, '纯滑窗完全看不到 sink（被窗口排除）'
assert sink_mass_stream > 0.5, 'StreamingLLM 保住了 sink（吸走大部分注意力）'
print('✅ 纯滑窗丢掉 sink → 模型失去泄压阀；StreamingLLM 保 sink → 注意力分布健康')

## 5 · block-sparse：局部块 + 全局列

块稀疏以**块**为单位决定算/不算（硬件友好）。构造一个「对角块(局部) + 第0列块(全局sink)」的块掩码，展开成元素掩码。

In [ ]:
def block_sparse_mask(n, block, n_local_blocks=1, global_block0=True):
    '''块稀疏：每个 query 块看 自己附近 n_local_blocks 个块 + 第0个块(全局)。'''
    nb = (n + block - 1) // block
    blk = np.zeros((nb, nb), bool)
    for qb in range(nb):
        for kb in range(nb):
            if kb <= qb and qb - kb < n_local_blocks:   # 局部(含因果)
                blk[qb, kb] = True
            if global_block0 and kb == 0 and kb <= qb:  # 全局第0块
                blk[qb, kb] = True
    # 展开成元素掩码 + 因果
    m = np.repeat(np.repeat(blk, block, 0), block, 1)[:n, :n]
    i = np.arange(n)[:, None]; j = np.arange(n)[None, :]
    return m & (j <= i)

n, block = 16, 4
bs = block_sparse_mask(n, block)
print('块稀疏每行连接数:', bs.sum(1))
assert bs[:, :block].any(axis=0).any(), '第0块(全局)应被保留'
# 靠后的 query 仍能看到第0块（全局通路），即使它在局部窗口外
assert bs[-1, 0] == True, '最后的 query 应能看到全局第0个 token'
assert bs.sum() < n*(n+1)//2, '块稀疏比因果全注意力稀疏'
print('✅ block-sparse：对角块=局部、第0列块=全局通路，块粒度对 GPU 友好')

## 6 · 召回率：稀疏漏掉了多少？

召回率 = 稀疏模式保留的注意力质量占全注意力的比例 = 全注意力 `P_full` 中被 mask 保留位置的概率和（每行和为1，故=保留位置概率之和）。
**关键场景**：针在窗口内召回≈1、针在窗口外召回≈0——这正是纯滑窗在长程检索上失败的原因。

In [ ]:
def attention_recall(P_full, mask):
    '''每个 query 的召回率 = 被 mask 保留位置的全注意力概率之和。'''
    kept = np.where(mask, P_full, 0.0)
    return kept.sum(axis=1)                       # (n,) 每行召回率

n, d, W = 32, 8, 6
Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
_, P_full = masked_attention(Q, K, V, np.ones((n,n), bool))
sw = sliding_window_mask(n, W)
recall = attention_recall(P_full, sw)
print(f'滑窗 W={W} 平均召回率: {recall.mean():.3f}')

# 「大海捞针」：把 query=n-1 对一个远处 key 的注意力做强（针在窗口外）
q = n - 1; needle = 2                             # 针在位置2，远在窗口外
S = Q @ K.T / np.sqrt(d); S[q, needle] += 20.0    # 让 q 强烈关注远处的针
P_needle = softmax_rows(S)
recall_in_window  = attention_recall(P_needle, sliding_window_mask(n, n))[q]  # 大窗口=包含针
recall_out_window = attention_recall(P_needle, sw)[q]                          # 小窗口=漏针
print(f'针在窗内召回={recall_in_window:.3f}  针在窗外召回={recall_out_window:.3f}')
assert recall_in_window > 0.9, '针在窗口内应召回≈1'
assert recall_out_window < 0.1, '针在窗口外应召回≈0（漏看）'
print('✅ 召回率量化稀疏损失：针在窗外被纯滑窗完全漏看 → NIAH 失败的根源')

---
## ✏️ 练习 1：实现因果滑动窗口掩码

实现 `swa_mask(n, W)`：因果滑窗，query i 看 `(i-W, i]`。返回 (n,n) bool。
验证：每行连接数 ≤ W、第 0 行只看自己、靠后行看满 W、是因果（不看未来）。

In [ ]:
def swa_mask(n, W):
    # TODO: i=arange(n)[:,None], j=arange(n)[None,:]; 返回 (j<=i)&(j>i-W)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
n, W = 10, 3
m = swa_mask(n, W)
assert m.shape == (n, n) and m.dtype == bool
assert m[0].sum() == 1, '第0行只看自己'
assert m[-1].sum() == W, '靠后行看满 W'
assert not m[np.triu_indices(n, k=1)].any(), '因果：不看未来'
assert m.sum() <= n * W, '总连接 <= n*W'
print('✅ 练习 1 通过：因果滑窗掩码正确，复杂度 O(n·W)')

## ✏️ 练习 2：给滑窗加 attention sink

实现 `sink_mask(n, W, n_sink)`：在因果滑窗基础上，**始终保留**最前 `n_sink` 个 token（sink）。
验证：靠后的 query 即使在局部窗口外，仍能看到 sink token。

In [ ]:
def sink_mask(n, W, n_sink):
    # TODO: 滑窗 (j<=i)&(j>i-W)  或  sink (j<n_sink)&(j<=i)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
n, W, n_sink = 32, 4, 2
m = sink_mask(n, W, n_sink)
assert m[n-1, 0] and m[n-1, 1], '最后的 query 应看到前 2 个 sink'
assert not m[n-1, n//2], '窗口外的非 sink token 不应被看到'
assert m[n-1, n-1], '应看到自己'
# sink 让靠后 query 多看到 n_sink 个连接
assert sink_mask(n, W, 2)[n-1].sum() > sliding_window_mask(n, W)[n-1].sum()
print('✅ 练习 2 通过：sink token 被始终保留 → StreamingLLM 的无限流关键')

## ✏️ 练习 3：block-sparse 掩码

实现 `block_diag_mask(n, block)`：只保留**对角块**（每个 query 块看同一个 key 块，含因果）。
验证：块内全连接、块间不连（除因果下三角内的对角块本身）、比全注意力稀疏。

In [ ]:
def block_diag_mask(n, block):
    # TODO: query i 和 key j 在同一个块(i//block==j//block) 且 因果(j<=i) 才保留
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
n, block = 12, 4
m = block_diag_mask(n, block)
assert m[0, 0] and m[3, 0] and m[3, 3], '同一对角块内(因果)应连通'
assert not m[4, 3], '跨块(块1的query看块0的key)不应连通'
assert not m[0, 3], '因果：不看未来'
assert m.sum() < n*(n+1)//2, '应比因果全注意力稀疏'
print('✅ 练习 3 通过：块对角掩码（块粒度，GPU 友好）')

## ✏️ 练习 4：评估稀疏模式的召回率

实现 `mean_recall(P_full, mask)`：返回所有 query 的平均召回率（被 mask 保留位置的全注意力概率之和的均值）。
验证：更大的窗口召回率更高；全 True 掩码召回率 = 1。

In [ ]:
def mean_recall(P_full, mask):
    # TODO: 返回 where(mask, P_full, 0).sum(axis=1).mean()
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
n, d = 24, 8
Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
_, P_full = masked_attention(Q, K, V, np.ones((n,n), bool))
r_full  = mean_recall(P_full, np.ones((n,n), bool))
r_big   = mean_recall(P_full, sliding_window_mask(n, 12))
r_small = mean_recall(P_full, sliding_window_mask(n, 3))
assert abs(r_full - 1.0) < 1e-9, '全 True 掩码召回率=1'
assert r_big > r_small, '更大窗口召回率更高'
print(f'召回率：全注意力={r_full:.3f}  W=12: {r_big:.3f}  W=3: {r_small:.3f}')
print('✅ 练习 4 通过：召回率把「稀疏化损失」变成可测量的工程权衡')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def swa_mask(n, W):
    i = np.arange(n)[:, None]; j = np.arange(n)[None, :]
    return (j <= i) & (j > i - W)

In [ ]:
# 练习 2 参考答案
def sink_mask(n, W, n_sink):
    i = np.arange(n)[:, None]; j = np.arange(n)[None, :]
    local = (j <= i) & (j > i - W)
    sink  = (j < n_sink) & (j <= i)
    return local | sink

In [ ]:
# 练习 3 参考答案
def block_diag_mask(n, block):
    i = np.arange(n)[:, None]; j = np.arange(n)[None, :]
    same_block = (i // block) == (j // block)
    return same_block & (j <= i)

In [ ]:
# 练习 4 参考答案
def mean_recall(P_full, mask):
    return np.where(mask, P_full, 0.0).sum(axis=1).mean()

---
## 🧪 真实数据胶囊：Mistral 滑窗能覆盖多长？

用 **Mistral-7B** 的真实配置（滑窗 W=4096，32 层）算它的理论感受野，对比它宣称的上下文长度，
并算稀疏注意力相对全注意力省了多少算力。

In [ ]:
def mistral_receptive_and_savings(n, W=4096, n_layers=32):
    # TODO: 返回 (感受野, 全注意力连接数, 滑窗连接数, 省的比例)
    #   感受野 = n_layers*(W-1)+1
    #   full = n*(n+1)//2 (因果)
    #   swa  = sum over i of min(i+1, W)  ≈ 每行 min(位置+1, W)
    #   saving = 1 - swa/full
    raise NotImplementedError

In [ ]:
# 自测
for n in [8192, 32768, 131072]:
    rf, full, swa, save = mistral_receptive_and_savings(n)
    print(f'n={n:>7d}: 感受野={rf:>7d}  滑窗省算力={save:>5.1%}  (全注意力连接 {full:.2e} → 滑窗 {swa:.2e})')
rf, full, swa, save = mistral_receptive_and_savings(131072)
assert rf > 100000, 'Mistral 32 层感受野应 >100k'
assert save > 0.9, '128k 下滑窗应省 >90% 算力'
print('\n✅ Mistral 滑窗：32 层感受野 ~131k 覆盖长上下文，且 128k 下省 >90% 注意力算力')
print('   代价：单层只看 4096，超出感受野的精确长程检索仍需全局 token 或与全注意力混合。')

In [ ]:
# 📖 胶囊参考答案
def mistral_receptive_and_savings(n, W=4096, n_layers=32):
    rf = n_layers * (W - 1) + 1
    full = n * (n + 1) // 2
    pos = np.arange(1, n + 1)
    swa = int(np.minimum(pos, W).sum())
    return rf, full, swa, 1 - swa / full

### 小结
- **算力墙**：FlashAttention 省显存但算力仍 O(n²)。稀疏注意力让每 query 只看部分 key → O(n·W)。
- **滑窗 SWA**：局部带状掩码，单层看 W、**L 层感受野≈L×W**（局部+深度=长程）。
- **attention sink**：softmax 必须分配满预算 → 模型把多余注意力倾倒到开头几个 token。**StreamingLLM** = sink + 滑窗 → 无限流。
- **block-sparse / Longformer / BigBird** = 局部块 + 全局 token + 随机块 的不同配方；块粒度对 GPU 友好。
- **召回率**量化稀疏损失（保留位置的全注意力概率和）；纯滑窗对**窗口外的针**召回≈0 → NIAH 失败的根源。
- 稀疏是**近似**：先确认 FlashAttention(精确) 不够用，再用召回率 + 真实评测(模块05) 确认损失可接受。

下一站：**模块 04 · 线性注意力与 SSM** —— 稀疏仍保留 softmax 形式；下一步是彻底换掉它，把复杂度降到 O(n)。